# Bien Bao + Bien So Xe Viet Nam - v7.3
## A. Bien bao: YOLO fine-tune + CLIP classify
## B. Bien so xe: YOLO fine-tune + EasyOCR

| Module | Detect | Classify/OCR | Output |
|---|---|---|---|
| **A** | YOLO (train tren data/full/archive) | CLIP fine-tune | BBox + P.102, W.224... |
| **B** | YOLO (train tren yolo_plate_dataset) | EasyOCR | BBox + "51F-123.45" |


In [1]:
# CELL 1 - Imports
import subprocess, sys
def pip_install(pkg):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])
pip_install('ultralytics>=8.2')
pip_install('easyocr')

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, TensorDataset
import cv2
import numpy as np
import open_clip
from ultralytics import YOLO
from PIL import Image, ImageEnhance, ImageFilter
from pathlib import Path
import os, random, warnings, time, copy, shutil, yaml, zipfile
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from collections import defaultdict
from typing import List, Tuple, Dict, Set, Optional
from sklearn.metrics import confusion_matrix, classification_report
import seaborn as sns
import easyocr

warnings.filterwarnings('ignore')
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
print(f'Device: {device}')

print('Loading EasyOCR...')
ocr_reader = easyocr.Reader(['en'], gpu=torch.cuda.is_available())
print('EasyOCR loaded (English - cho bien so xe VN)')

GPU: NVIDIA GeForce RTX 3070
VRAM: 8.6 GB
Device: cuda
Loading EasyOCR...
EasyOCR loaded (English - cho bien so xe VN)


In [2]:
# CELL 2 - Chuan bi data cho YOLO fine-tune (FIXED data leak)
print('=' * 60)
print('BUOC 1: CHUAN BI DATA CHO YOLO FINE-TUNE')
print('=' * 60)

ARCHIVE_DIR  = Path('data/full/archive')
IMAGES_DIR   = ARCHIVE_DIR / 'images'
LABELS_DIR   = ARCHIVE_DIR / 'labels'
SPLIT_DIR    = ARCHIVE_DIR / 'split_dataset'

# Doc classes
classes_file = ARCHIVE_DIR / 'classes.txt'
if not classes_file.exists():
    classes_file = ARCHIVE_DIR / 'classes_en.txt'
with open(classes_file, 'r', encoding='utf-8') as f:
    yolo_classes = [line.strip() for line in f if line.strip()]
print(f'YOLO classes: {len(yolo_classes)}')

all_images = sorted(list(IMAGES_DIR.glob('*.jpg')) + list(IMAGES_DIR.glob('*.png'))
                    + list(IMAGES_DIR.glob('*.jpeg')))
all_labels = sorted(list(LABELS_DIR.glob('*.txt')))
print(f'Images: {len(all_images)} | Labels: {len(all_labels)}')

# XOA SAP yolo_dataset cu de tranh data leak khi chay lai
YOLO_DATA_DIR = Path('yolo_dataset')
if YOLO_DATA_DIR.exists():
    shutil.rmtree(YOLO_DATA_DIR)
for split in ['train', 'val', 'test']:
    (YOLO_DATA_DIR / 'images' / split).mkdir(parents=True, exist_ok=True)
    (YOLO_DATA_DIR / 'labels' / split).mkdir(parents=True, exist_ok=True)

# Check split_dataset co san
if SPLIT_DIR.exists() and (SPLIT_DIR / 'train').exists():
    print('\nTim thay split_dataset co san...')
    for split in ['train', 'val', 'test']:
        src_img = SPLIT_DIR / split / 'images'
        src_lbl = SPLIT_DIR / split / 'labels'
        if not src_img.exists(): continue
        for img_f in src_img.glob('*'):
            if img_f.suffix.lower() in ['.jpg', '.png', '.jpeg']:
                shutil.copy2(img_f, YOLO_DATA_DIR / 'images' / split / img_f.name)
                lbl_f = src_lbl / (img_f.stem + '.txt')
                if lbl_f.exists():
                    shutil.copy2(lbl_f, YOLO_DATA_DIR / 'labels' / split / lbl_f.name)
    # Neu khong co test rieng -> YOLO chi can train+val
    # Merge test vao val cho YOLO (YOLO chi can 2 split)
    test_img_dir = YOLO_DATA_DIR / 'images' / 'test'
    if test_img_dir.exists() and list(test_img_dir.glob('*')):
        print('  (merge test -> val for YOLO training)')
        for f in test_img_dir.glob('*'):
            shutil.move(str(f), YOLO_DATA_DIR / 'images' / 'val' / f.name)
        for f in (YOLO_DATA_DIR / 'labels' / 'test').glob('*'):
            shutil.move(str(f), YOLO_DATA_DIR / 'labels' / 'val' / f.name)
else:
    print('\nTu dong split 80/20...')
    pairs = []
    for img in all_images:
        lbl = LABELS_DIR / (img.stem + '.txt')
        if lbl.exists():
            pairs.append((img, lbl))
    random.seed(SEED)
    random.shuffle(pairs)
    n_train = int(len(pairs) * 0.8)
    for img, lbl in pairs[:n_train]:
        shutil.copy2(img, YOLO_DATA_DIR / 'images' / 'train' / img.name)
        shutil.copy2(lbl, YOLO_DATA_DIR / 'labels' / 'train' / lbl.name)
    for img, lbl in pairs[n_train:]:
        shutil.copy2(img, YOLO_DATA_DIR / 'images' / 'val' / img.name)
        shutil.copy2(lbl, YOLO_DATA_DIR / 'labels' / 'val' / lbl.name)

n_train = len(list((YOLO_DATA_DIR / 'images' / 'train').glob('*')))
n_val = len(list((YOLO_DATA_DIR / 'images' / 'val').glob('*')))
print(f'\nYOLO dataset: Train={n_train} | Val={n_val} | Total={n_train+n_val}')
assert n_train + n_val <= len(all_images) + 100, f'Data leak! {n_train}+{n_val} > {len(all_images)}'

# data.yaml
data_yaml = {
    'path': str(YOLO_DATA_DIR.resolve()),
    'train': 'images/train',
    'val': 'images/val',
    'nc': len(yolo_classes),
    'names': yolo_classes,
}
yaml_path = YOLO_DATA_DIR / 'data.yaml'
with open(yaml_path, 'w', encoding='utf-8') as f:
    yaml.dump(data_yaml, f, default_flow_style=False, allow_unicode=True)
print(f'data.yaml saved: {yaml_path}')

BUOC 1: CHUAN BI DATA CHO YOLO FINE-TUNE
YOLO classes: 52
Images: 3216 | Labels: 3216

Tu dong split 80/20...

YOLO dataset: Train=2572 | Val=644 | Total=3216
data.yaml saved: yolo_dataset\data.yaml


In [4]:
# ====================== CELL 3 MỚI - FINE-TUNE YOLOv8s (ĐÃ SỬA) ======================
print('=' * 60)
print('MODULE A: FINE-TUNE YOLOv8s (UPGRADED)')
print('  - YOLOv8s: 11.2M params')
print('  - imgsz=960: detect biển nhỏ tốt hơn')
print('  - multi_scale=0.5 (an toàn)')
print('=' * 60)

yolo_model = YOLO('yolov8s.pt')

YOLO_EPOCHS = 100
YOLO_IMGSZ  = 960
YOLO_BATCH  = 4

print(f'Model: YOLOv8s (11.2M params)')
print(f'Config: epochs={YOLO_EPOCHS}, imgsz={YOLO_IMGSZ}, batch={YOLO_BATCH}')

results = yolo_model.train(
    data=str((YOLO_DATA_DIR / 'data.yaml').resolve()),
    epochs=YOLO_EPOCHS,
    imgsz=YOLO_IMGSZ,
    batch=YOLO_BATCH,
    device=0,
    workers=0,
    patience=20,
    save=True,
    project='runs/detect',
    name='traffic_signs_vn_v2',
    exist_ok=True,

    # Augmentation (giữ nguyên, chỉ sửa multi_scale)
    hsv_h=0.015,
    hsv_s=0.5,
    hsv_v=0.4,
    degrees=15,
    translate=0.15,
    scale=0.5,
    shear=2.0,
    perspective=0.0005,
    flipud=0.0,
    fliplr=0.5,
    mosaic=1.0,
    mixup=0.15,
    copy_paste=0.1,
    erasing=0.3,

    # Training config
    close_mosaic=20,
    multi_scale=0.5,           # ← SỬA Ở ĐÂY (float 0.5 thay vì True)
    optimizer='AdamW',
    lr0=0.001,
    lrf=0.01,
    cos_lr=True,
    weight_decay=0.0005,
    warmup_epochs=5,
    amp=True,
    cache=False,

    box=7.5,
    cls=0.5,
    dfl=1.5,
)

print('\n>>> YOLO Bien Bao v2 (YOLOv8s) HOAN TAT!')

MODULE A: FINE-TUNE YOLOv8s (UPGRADED)
  - YOLOv8s: 11.2M params
  - imgsz=960: detect biển nhỏ tốt hơn
  - multi_scale=0.5 (an toàn)
Model: YOLOv8s (11.2M params)
Config: epochs=100, imgsz=960, batch=4
New https://pypi.org/project/ultralytics/8.4.36 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.33  Python-3.11.9 torch-2.6.0+cu124 CUDA:0 (NVIDIA GeForce RTX 3070, 8192MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=4, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=20, cls=0.5, compile=False, conf=None, copy_paste=0.1, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=C:\ n\TrafficSignVN\yolo_dataset\data.yaml, degrees=15, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.3, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.5, hsv_v=0.4, img

In [5]:
# CELL 4 - Danh gia YOLO bien bao (voi TTA)
print('=' * 60)
print('DANH GIA YOLO BIEN BAO (YOLOv8s + TTA)')
print('=' * 60)

best_yolo_path = None
for pattern in ['traffic_signs_vn_v2', 'traffic_signs_vn']:
    for p in sorted(Path('runs/detect').rglob('best.pt')):
        if pattern in str(p):
            best_yolo_path = p; break
    if best_yolo_path: break
if best_yolo_path is None:
    candidates = list(Path('runs').rglob('best.pt'))
    if candidates: best_yolo_path = sorted(candidates)[-1]

print(f'Loaded: {best_yolo_path}')
yolo_finetuned = YOLO(str(best_yolo_path))

# Eval KHONG TTA (baseline)
print('\n--- Without TTA ---')
metrics_no_tta = yolo_finetuned.val(
    data=str((YOLO_DATA_DIR / 'data.yaml').resolve()),
    workers=0, verbose=False)
print(f'  mAP50:    {metrics_no_tta.box.map50:.4f}')
print(f'  mAP50-95: {metrics_no_tta.box.map:.4f}')

# Eval VOI TTA (augment=True)
print('\n--- With TTA (Test-Time Augmentation) ---')
metrics = yolo_finetuned.val(
    data=str((YOLO_DATA_DIR / 'data.yaml').resolve()),
    workers=0, verbose=True,
    augment=True)           # TTA: flip + multi-scale at test time
print(f'  mAP50:    {metrics.box.map50:.4f}')
print(f'  mAP50-95: {metrics.box.map:.4f}')

improvement = (metrics.box.map - metrics_no_tta.box.map) * 100
print(f'\n  TTA improvement: +{improvement:.2f}% mAP50-95')

# Training curves
for p in sorted(Path('runs/detect').rglob('results.png'), reverse=True):
    if 'v2' in str(p) or 'traffic_signs' in str(p):
        img = plt.imread(str(p))
        fig, ax = plt.subplots(figsize=(16, 8))
        ax.imshow(img); ax.axis('off')
        ax.set_title('YOLOv8s Training Results', fontsize=14)
        plt.tight_layout(); plt.show()
        break

DANH GIA YOLO BIEN BAO (YOLOv8s + TTA)
Loaded: runs\detect\runs\detect\traffic_signs_vn_v2\weights\best.pt

--- Without TTA ---
Ultralytics 8.4.33  Python-3.11.9 torch-2.6.0+cu124 CUDA:0 (NVIDIA GeForce RTX 3070, 8192MiB)
Model summary (fused): 73 layers, 11,145,708 parameters, 0 gradients, 28.5 GFLOPs
val: Fast image access  (ping: 0.00.0 ms, read: 2369.0629.3 MB/s, size: 179.9 KB)
val: Scanning C:\Đồ án\TrafficSignVN\yolo_dataset\labels\val.cache... 644 images, 5 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 644/644  0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 41/41 4.5it/s 9.1s0.3s
                   all        644       1649      0.966      0.973      0.989      0.772
Speed: 0.3ms preprocess, 5.5ms inference, 0.0ms loss, 0.5ms postprocess per image
Results saved to C:\ n\TrafficSignVN\runs\detect\val5
  mAP50:    0.9895
  mAP50-95: 0.7722

--- With TTA (Test-Time Augmentation) ---
Ultralytics 8.4.33  Python-3.11.9 tor

<Figure size 1600x800 with 1 Axes>

In [6]:
# CELL 5 - Test YOLO: Hien thi BOUNDING BOXES
print('=' * 60)
print('TEST YOLO DETECTION - BOUNDING BOXES')
print('=' * 60)

# Tim anh test
test_images = []
# Tu data/full/archive/images hoac split_dataset/test
for search_dir in [ARCHIVE_DIR / 'split_dataset' / 'test' / 'images',
                    ARCHIVE_DIR / 'images']:
    if search_dir.exists():
        imgs = list(search_dir.glob('*.jpg')) + list(search_dir.glob('*.png'))
        if imgs:
            test_images = imgs
            break

if not test_images:
    test_images = list(Path('data/full').rglob('*.jpg'))[:10]

random.seed(42)
sample = random.sample(test_images, min(6, len(test_images)))
print(f'Testing on {len(sample)} images...\n')

fig, axes = plt.subplots(2, 3, figsize=(18, 12))
axes = axes.flatten()

for idx, img_path in enumerate(sample):
    if idx >= 6: break
    
    # Run YOLO detection
    results = yolo_finetuned(str(img_path), conf=0.3, verbose=False)
    
    # Draw results
    img_bgr = cv2.imread(str(img_path))
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    
    boxes = results[0].boxes
    n_detections = 0
    if boxes is not None and len(boxes) > 0:
        for j in range(len(boxes)):
            x1, y1, x2, y2 = boxes.xyxy[j].cpu().numpy().astype(int)
            conf = float(boxes.conf[j])
            cls_id = int(boxes.cls[j])
            cls_name = yolo_classes[cls_id] if cls_id < len(yolo_classes) else f'cls_{cls_id}'
            
            # Ve bounding box
            color = (0, 255, 0)
            cv2.rectangle(img_rgb, (x1, y1), (x2, y2), color, 3)
            
            # Ve label
            label = f'{cls_name} {conf:.2f}'
            (tw, th), _ = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 0.6, 2)
            cv2.rectangle(img_rgb, (x1, y1-th-10), (x1+tw, y1), color, -1)
            cv2.putText(img_rgb, label, (x1, y1-5),
                       cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0,0,0), 2)
            n_detections += 1
    
    axes[idx].imshow(img_rgb)
    axes[idx].set_title(f'{img_path.name}: {n_detections} detections', fontsize=10)
    axes[idx].axis('off')

plt.suptitle('YOLO Fine-tuned: Bounding Box Detection', fontsize=16)
plt.tight_layout()
plt.show()
print('>>> Bounding boxes hien thi thanh cong!')

TEST YOLO DETECTION - BOUNDING BOXES
Testing on 6 images...



<Figure size 1800x1200 with 6 Axes>

>>> Bounding boxes hien thi thanh cong!


In [7]:
# CELL 6 - Load CLIP + Train classifier tren data/classes
print('=' * 60)
print('BUOC 3: TRAIN CLIP CLASSIFIER')
print('=' * 60)

# Load CLIP
CLIP_MODEL    = 'ViT-L-14'
CLIP_PRETRAIN = 'laion2b_s32b_b82k'
CLIP_DIM      = 768

model_clip, _, preprocess_clip = open_clip.create_model_and_transforms(
    CLIP_MODEL, pretrained=CLIP_PRETRAIN)
model_clip = model_clip.to(device).eval()
tokenizer = open_clip.get_tokenizer(CLIP_MODEL)
print(f'CLIP {CLIP_MODEL} loaded')

# Data split 70/15/15 tu data/classes
DATA_CLASSES = Path('data/classes')
SKIP = {'Camera'}
MIN_IMGS = 5

class_dirs = sorted([d for d in DATA_CLASSES.iterdir()
                      if d.is_dir() and d.name not in SKIP])

train_data, val_data, test_data = [], [], []
class_names_list = []
class_to_idx = {}

random.seed(SEED)
for cd in class_dirs:
    cn = cd.name
    files = sorted(list(cd.glob('*.jpg')) + list(cd.glob('*.png')) + list(cd.glob('*.jpeg')))
    if len(files) < MIN_IMGS: continue
    
    class_to_idx[cn] = len(class_names_list)
    class_names_list.append(cn)
    
    shuffled = files.copy()
    random.shuffle(shuffled)
    n = len(shuffled)
    n_train = max(3, int(n * 0.70))
    n_val = max(1, int(n * 0.15))
    
    train_data.extend([(str(f), cn) for f in shuffled[:n_train]])
    val_data.extend([(str(f), cn) for f in shuffled[n_train:n_train+n_val]])
    test_data.extend([(str(f), cn) for f in shuffled[n_train+n_val:]])

NUM_CLASSES = len(class_names_list)
print(f'Classes: {NUM_CLASSES}')
print(f'Train: {len(train_data)} | Val: {len(val_data)} | Test: {len(test_data)}')

# Dataset
class SignDataset(Dataset):
    def __init__(self, data, c2i, transform, augment=False):
        self.data, self.c2i, self.transform, self.augment = data, c2i, transform, augment
    def __len__(self): return len(self.data)
    def __getitem__(self, idx):
        path, cn = self.data[idx]
        img = Image.open(path).convert('RGB')
        if self.augment:
            if random.random()>0.5: img = img.rotate(random.uniform(-15,15), fillcolor=(128,128,128))
            if random.random()>0.5: img = ImageEnhance.Brightness(img).enhance(random.uniform(0.7,1.3))
            if random.random()>0.5: img = ImageEnhance.Contrast(img).enhance(random.uniform(0.8,1.2))
            if random.random()>0.5: img = ImageEnhance.Color(img).enhance(random.uniform(0.7,1.3))
        return self.transform(img), self.c2i[cn]

BS = 32
train_loader = DataLoader(SignDataset(train_data, class_to_idx, preprocess_clip, True),
                          batch_size=BS, shuffle=True, num_workers=0, pin_memory=True)
val_loader = DataLoader(SignDataset(val_data, class_to_idx, preprocess_clip, False),
                        batch_size=BS, shuffle=False, num_workers=0, pin_memory=True)
test_loader = DataLoader(SignDataset(test_data, class_to_idx, preprocess_clip, False),
                         batch_size=BS, shuffle=False, num_workers=0, pin_memory=True)
print(f'DataLoaders ready (num_workers=0 for Windows)')

BUOC 3: TRAIN CLIP CLASSIFIER
CLIP ViT-L-14 loaded
Classes: 38
Train: 3668 | Val: 771 | Test: 823
DataLoaders ready (num_workers=0 for Windows)


In [8]:
# CELL 7 - Train CLIP Classifier (MLP Head + Fine-tune)

# MLP Head
class CLIPClassifier(nn.Module):
    def __init__(self, clip_model, clip_dim, num_classes, freeze_backbone=True):
        super().__init__()
        self.clip = clip_model
        self.freeze_backbone = freeze_backbone
        if freeze_backbone:
            for p in self.clip.parameters(): p.requires_grad = False
        self.head = nn.Sequential(
            nn.Linear(clip_dim, 512), nn.BatchNorm1d(512), nn.ReLU(True), nn.Dropout(0.3),
            nn.Linear(512, 256), nn.BatchNorm1d(256), nn.ReLU(True), nn.Dropout(0.2),
            nn.Linear(256, num_classes))
        for m in self.head.modules():
            if isinstance(m, nn.Linear):
                nn.init.kaiming_normal_(m.weight, mode='fan_out')
                nn.init.zeros_(m.bias)
    
    def forward(self, x):
        with torch.no_grad() if self.freeze_backbone else torch.enable_grad():
            f = self.clip.encode_image(x)
            f = f / f.norm(dim=-1, keepdim=True)
        if self.freeze_backbone: f = f.float()
        return self.head(f)

classifier = CLIPClassifier(model_clip, CLIP_DIM, NUM_CLASSES, freeze_backbone=True).to(device)
print(f'Trainable: {sum(p.numel() for p in classifier.parameters() if p.requires_grad):,} params')

# ===== Phase 2: Train MLP Head (feature caching) =====
print('\n--- Phase 2: Train MLP Head (frozen CLIP) ---')

def extract_feats(loader):
    model_clip.eval()
    feats, labels = [], []
    with torch.no_grad():
        for imgs, lbs in loader:
            f = model_clip.encode_image(imgs.to(device))
            f = f / f.norm(dim=-1, keepdim=True)
            feats.append(f.cpu()); labels.append(lbs)
    return torch.cat(feats), torch.cat(labels)

train_f, train_l = extract_feats(train_loader)
val_f, val_l = extract_feats(val_loader)
test_f, test_l = extract_feats(test_loader)
print(f'Features: train={train_f.shape} val={val_f.shape} test={test_f.shape}')

fast_train = DataLoader(TensorDataset(train_f, train_l), batch_size=1024, shuffle=True, num_workers=0)
fast_val = DataLoader(TensorDataset(val_f, val_l), batch_size=1024, shuffle=False, num_workers=0)

criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
optimizer = optim.AdamW(classifier.head.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=30, eta_min=1e-6)

best_va, best_st, hist = 0, None, {'tl':[],'vl':[],'ta':[],'va':[]}

for ep in range(1, 31):
    classifier.head.train()
    tl, tc, tt = 0, 0, 0
    for f, l in fast_train:
        f, l = f.to(device), l.to(device)
        optimizer.zero_grad()
        loss = criterion(classifier.head(f), l)
        loss.backward(); optimizer.step()
        tl += loss.item()*len(l); tc += (classifier.head(f).argmax(1)==l).sum().item(); tt += len(l)
    
    classifier.head.eval()
    vl, vc, vt = 0, 0, 0
    with torch.no_grad():
        for f, l in fast_val:
            f, l = f.to(device), l.to(device)
            lo = criterion(classifier.head(f), l)
            vl += lo.item()*len(l); vc += (classifier.head(f).argmax(1)==l).sum().item(); vt += len(l)
    
    hist['tl'].append(tl/tt); hist['vl'].append(vl/vt)
    hist['ta'].append(tc/tt); hist['va'].append(vc/vt)
    scheduler.step()
    
    mk = ''
    if vc/vt > best_va:
        best_va = vc/vt; best_st = copy.deepcopy(classifier.head.state_dict()); mk = ' *BEST*'
    if ep % 5 == 0 or ep == 1 or mk:
        print(f'  Ep {ep:2d}/30 | Train: {tc/tt*100:.1f}% | Val: {vc/vt*100:.1f}%{mk}')

classifier.head.load_state_dict(best_st)

# Test Phase 2
classifier.head.eval()
p2c = 0
with torch.no_grad():
    for f, l in DataLoader(TensorDataset(test_f, test_l), batch_size=1024, num_workers=0):
        f, l = f.to(device), l.to(device)
        p2c += (classifier.head(f).argmax(1)==l).sum().item()
p2_acc = p2c / len(test_l)
print(f'\n>>> Phase 2 Test Accuracy: {p2_acc*100:.1f}%')

# ===== Phase 3: Fine-tune CLIP backbone =====
print('\n--- Phase 3: Fine-tune CLIP backbone ---')
classifier.freeze_backbone = False
for p in classifier.clip.parameters(): p.requires_grad = False

# Unfreeze last 4 blocks
visual = classifier.clip.visual
if hasattr(visual, 'transformer'):
    blocks = visual.transformer.resblocks
elif hasattr(visual, 'trunk'):
    blocks = visual.trunk.blocks
else:
    blocks = []
for i in range(max(0, len(blocks)-4), len(blocks)):
    for p in blocks[i].parameters(): p.requires_grad = True
if hasattr(visual, 'ln_post'):
    for p in visual.ln_post.parameters(): p.requires_grad = True
if hasattr(visual, 'proj') and visual.proj is not None:
    if isinstance(visual.proj, nn.Parameter): visual.proj.requires_grad = True

for p in classifier.head.parameters(): p.requires_grad = True

tp3 = sum(p.numel() for p in classifier.parameters() if p.requires_grad)
print(f'Trainable: {tp3:,} params')

bb_p = [p for n,p in classifier.named_parameters() if p.requires_grad and 'head' not in n]
hd_p = [p for n,p in classifier.named_parameters() if p.requires_grad and 'head' in n]
opt3 = optim.AdamW([{'params':bb_p,'lr':1e-6},{'params':hd_p,'lr':5e-5}], weight_decay=1e-4)
sch3 = optim.lr_scheduler.CosineAnnealingLR(opt3, T_max=15, eta_min=1e-7)

best_va3 = best_va
best_st3 = copy.deepcopy(classifier.state_dict())

for ep in range(1, 16):
    t0 = time.time()
    classifier.train()
    tl, tc, tt = 0, 0, 0
    for imgs, lbs in train_loader:
        imgs, lbs = imgs.to(device), lbs.to(device)
        opt3.zero_grad()
        logits = classifier(imgs)
        loss = criterion(logits, lbs)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(classifier.parameters(), 1.0)
        opt3.step()
        tl += loss.item()*len(lbs); tc += (logits.argmax(1)==lbs).sum().item(); tt += len(lbs)
    
    classifier.eval()
    vl, vc, vt = 0, 0, 0
    with torch.no_grad():
        for imgs, lbs in val_loader:
            imgs, lbs = imgs.to(device), lbs.to(device)
            logits = classifier(imgs)
            lo = criterion(logits, lbs)
            vl += lo.item()*len(lbs); vc += (logits.argmax(1)==lbs).sum().item(); vt += len(lbs)
    
    sch3.step()
    dt = time.time() - t0
    mk = ''
    if vc/vt > best_va3:
        best_va3 = vc/vt; best_st3 = copy.deepcopy(classifier.state_dict()); mk = ' *BEST*'
    if ep % 3 == 0 or ep == 1 or mk:
        print(f'  Ep {ep:2d}/15 | Train: {tc/tt*100:.1f}% | Val: {vc/vt*100:.1f}% | {dt:.0f}s{mk}')

classifier.load_state_dict(best_st3)

# Test Phase 3
classifier.eval()
p3c, p3t = 0, 0
p3_yt, p3_yp = [], []
with torch.no_grad():
    for imgs, lbs in test_loader:
        imgs, lbs = imgs.to(device), lbs.to(device)
        preds = classifier(imgs).argmax(1)
        p3c += (preds==lbs).sum().item(); p3t += len(lbs)
        p3_yt.extend(lbs.cpu().numpy().tolist())
        p3_yp.extend(preds.cpu().numpy().tolist())
p3_acc = p3c / p3t
print(f'\n>>> Phase 3 Test Accuracy: {p3_acc*100:.1f}%')
print(f'    Phase 2: {p2_acc*100:.1f}% -> Phase 3: {p3_acc*100:.1f}% (+{(p3_acc-p2_acc)*100:.1f}%)')

Trainable: 536,358 params

--- Phase 2: Train MLP Head (frozen CLIP) ---
Features: train=torch.Size([3668, 768]) val=torch.Size([771, 768]) test=torch.Size([823, 768])
  Ep  1/30 | Train: 22.6% | Val: 20.6% *BEST*
  Ep  2/30 | Train: 53.9% | Val: 35.9% *BEST*
  Ep  3/30 | Train: 66.8% | Val: 49.2% *BEST*
  Ep  4/30 | Train: 74.2% | Val: 62.8% *BEST*
  Ep  5/30 | Train: 77.9% | Val: 78.5% *BEST*
  Ep  6/30 | Train: 81.9% | Val: 88.8% *BEST*
  Ep  7/30 | Train: 84.4% | Val: 91.1% *BEST*
  Ep  8/30 | Train: 86.0% | Val: 92.3% *BEST*
  Ep  9/30 | Train: 86.9% | Val: 92.5% *BEST*
  Ep 10/30 | Train: 88.4% | Val: 92.6% *BEST*
  Ep 11/30 | Train: 88.4% | Val: 93.3% *BEST*
  Ep 12/30 | Train: 89.4% | Val: 94.2% *BEST*
  Ep 13/30 | Train: 90.8% | Val: 94.7% *BEST*
  Ep 14/30 | Train: 91.2% | Val: 95.1% *BEST*
  Ep 15/30 | Train: 91.4% | Val: 94.9%
  Ep 17/30 | Train: 91.7% | Val: 95.2% *BEST*
  Ep 18/30 | Train: 92.5% | Val: 95.5% *BEST*
  Ep 19/30 | Train: 92.5% | Val: 95.6% *BEST*
  Ep 20/30 

In [9]:
# CELL 8 - Training Curves + Classification Report

# Training curves
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
ax1.plot(hist['tl'], label='Train'); ax1.plot(hist['vl'], label='Val')
ax1.set_title('Phase 2: Loss'); ax1.legend(); ax1.grid(True, alpha=0.3)
ax2.plot([a*100 for a in hist['ta']], label='Train')
ax2.plot([a*100 for a in hist['va']], label='Val')
ax2.set_title('Phase 2: Accuracy'); ax2.legend(); ax2.grid(True, alpha=0.3)
plt.suptitle('CLIP Classifier Training', fontsize=14)
plt.tight_layout(); plt.show()

# Classification report
present = sorted(set(p3_yt))
names = [class_names_list[i] for i in present]
print('Classification Report (Phase 3 - CLIP Classifier):')
print(classification_report(p3_yt, p3_yp, labels=present, target_names=names, zero_division=0))

# Confusion matrix
if len(present) <= 30:
    cm = confusion_matrix(p3_yt, p3_yp, labels=present)
    fig, ax = plt.subplots(figsize=(max(10,len(present)*0.5), max(8,len(present)*0.4)))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=names, yticklabels=names, ax=ax)
    ax.set_xlabel('Predicted'); ax.set_ylabel('True'); ax.set_title('Confusion Matrix')
    plt.xticks(rotation=45, ha='right', fontsize=7); plt.yticks(fontsize=7)
    plt.tight_layout(); plt.show()

<Figure size 1400x500 with 2 Axes>

Classification Report (Phase 3 - CLIP Classifier):
              precision    recall  f1-score   support

        B.8a       0.92      0.92      0.92        13
      I.407a       0.97      1.00      0.98        29
       I.409       1.00      0.40      0.57         5
      I.434a       1.00      1.00      1.00        32
       P.102       1.00      0.98      0.99        52
      P.103a       1.00      1.00      1.00         4
       P.104       1.00      1.00      1.00         8
       P.107       0.93      0.93      0.93        14
       P.111       0.98      0.98      0.98        56
      P.123a       1.00      1.00      1.00        38
      P.123b       0.96      1.00      0.98        26
     P.124b1       1.00      1.00      1.00         3
      P.124c       1.00      0.97      0.99        34
      P.124d       1.00      1.00      1.00         3
       P.129       1.00      1.00      1.00         3
       P.130       0.98      0.98      0.98       126
      P.131a       0.96      0

In [10]:
# CELL 9 - FULL PIPELINE v7.1: YOLO + CLIP + Smart Fusion + BBox
print('=' * 60)
print('FULL PIPELINE v7.1')
print('  - YOLO detect -> Bounding Box')
print('  - CLIP classify -> Label')
print('  - Smart fusion khi YOLO va CLIP bat dong')
print('  - Filter bbox qua nho')
print('=' * 60)

# Map YOLO class -> CLIP class (nhieu class trung ten)
# Tao mapping tu dong dua tren ten class
yolo_to_clip = {}
for yc in yolo_classes:
    if yc in class_names_list:
        yolo_to_clip[yc] = yc  # exact match
    else:
        # Tim match gan nhat (P.127*50 -> khong co trong CLIP)
        yolo_to_clip[yc] = None

n_mapped = sum(1 for v in yolo_to_clip.values() if v is not None)
print(f'Class mapping: {n_mapped}/{len(yolo_classes)} YOLO classes co trong CLIP')
unmapped = [k for k, v in yolo_to_clip.items() if v is None]
if unmapped:
    print(f'  YOLO-only classes (CLIP khong biet): {unmapped[:10]}...')

# Min bbox size (pixel) - bbox nho qua thi CLIP khong classify duoc
MIN_BBOX_SIZE = 25  # pixels

@torch.no_grad()
def detect_and_classify_v71(image_path, yolo_model, clip_classifier,
                            preprocess, class_names, device,
                            conf=0.25, top_k=3):
    clip_classifier.eval()
    image_bgr = cv2.imread(str(image_path))
    if image_bgr is None:
        return [], image_bgr

    results = yolo_model(image_bgr, conf=conf, verbose=False)
    boxes = results[0].boxes

    detections = []
    if boxes is not None and len(boxes) > 0:
        for j in range(len(boxes)):
            x1, y1, x2, y2 = boxes.xyxy[j].cpu().numpy().astype(int)
            yolo_conf = float(boxes.conf[j])
            yolo_cls = int(boxes.cls[j])
            yolo_name = yolo_classes[yolo_cls] if yolo_cls < len(yolo_classes) else f'cls_{yolo_cls}'

            bw, bh = x2 - x1, y2 - y1

            # Filter bbox qua nho
            if bw < MIN_BBOX_SIZE or bh < MIN_BBOX_SIZE:
                continue

            # Crop ROI voi padding ty le theo kich thuoc bbox
            h, w = image_bgr.shape[:2]
            pad_x = max(5, int(bw * 0.1))
            pad_y = max(5, int(bh * 0.1))
            cx1, cy1 = max(0, x1 - pad_x), max(0, y1 - pad_y)
            cx2, cy2 = min(w, x2 + pad_x), min(h, y2 + pad_y)
            roi = image_bgr[cy1:cy2, cx1:cx2]

            if roi.shape[0] < 15 or roi.shape[1] < 15:
                continue

            # CLIP classify
            roi_pil = Image.fromarray(cv2.cvtColor(roi, cv2.COLOR_BGR2RGB))
            tensor = preprocess(roi_pil).unsqueeze(0).to(device)
            logits = clip_classifier(tensor)
            probs = torch.softmax(logits, dim=1)[0]
            topk_vals, topk_idx = probs.topk(min(top_k, len(class_names)))

            clip_label = class_names[topk_idx[0]]
            clip_score = float(topk_vals[0])

            # ===== SMART LABEL FUSION =====
            # Chon label cuoi cung dua tren do tin cay
            final_label = clip_label
            final_score = clip_score
            method = 'clip'

            yolo_in_clip = yolo_to_clip.get(yolo_name)

            if clip_score >= 0.5:
                # CLIP tu tin -> dung CLIP
                final_label = clip_label
                final_score = clip_score
                method = 'clip'
            elif yolo_in_clip is not None and yolo_conf >= 0.7:
                # CLIP khong tu tin nhung YOLO rat tu tin va class co trong CLIP
                # Kiem tra YOLO class co trong CLIP top-3 khong
                clip_top3_labels = [class_names[idx] for idx in topk_idx]
                if yolo_in_clip in clip_top3_labels:
                    final_label = yolo_in_clip
                    final_score = yolo_conf
                    method = 'yolo+clip_agree'
                else:
                    # YOLO va CLIP bat dong -> dung YOLO neu conf cao
                    final_label = yolo_in_clip
                    final_score = yolo_conf * 0.8  # giam score vi bat dong
                    method = 'yolo_override'
            elif yolo_in_clip is None and yolo_conf >= 0.6:
                # YOLO class khong co trong CLIP -> dung YOLO label truc tiep
                final_label = yolo_name
                final_score = yolo_conf
                method = 'yolo_only'
            else:
                # Ca hai deu khong tu tin
                final_label = clip_label
                final_score = clip_score
                method = 'clip_low'

            detections.append({
                'bbox': [int(x1), int(y1), int(x2), int(y2)],
                'bbox_size': (bw, bh),
                'yolo_conf': yolo_conf,
                'yolo_class': yolo_name,
                'clip_label': clip_label,
                'clip_score': clip_score,
                'clip_top3': [(class_names[idx], float(val)) for val, idx in zip(topk_vals, topk_idx)],
                'final_label': final_label,
                'final_score': final_score,
                'method': method,
                'roi': roi,
            })

    return detections, image_bgr


def visualize_v71(image_path, detections, image_bgr):
    img_rgb = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB).copy()
    n_det = len(detections)

    # Ve tat ca bounding boxes
    for i, det in enumerate(detections):
        x1, y1, x2, y2 = det['bbox']
        color = (0, 255, 0) if det['final_score'] >= 0.5 else (255, 165, 0)

        cv2.rectangle(img_rgb, (x1, y1), (x2, y2), color, 3)
        label = f"{det['final_label']} {det['final_score']:.2f}"
        (tw, th), _ = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 0.6, 2)
        cv2.rectangle(img_rgb, (x1, max(0,y1-th-8)), (x1+tw+4, y1), color, -1)
        cv2.putText(img_rgb, label, (x1+2, max(th+2, y1-3)),
                   cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0,0,0), 2)

    # Layout: anh goc (lon) + toi da 3 ROI
    n_roi = min(n_det, 3)
    ncols = 1 + n_roi
    fig, axes = plt.subplots(1, ncols, figsize=(5*ncols, 5),
                              gridspec_kw={'width_ratios': [2] + [1]*n_roi} if n_roi > 0 else None)
    if ncols == 1: axes = [axes]

    axes[0].imshow(img_rgb)
    axes[0].set_title(f'{Path(image_path).name} | {n_det} sign(s)', fontsize=11)
    axes[0].axis('off')

    for i in range(n_roi):
        det = detections[i]
        roi_rgb = cv2.cvtColor(det['roi'], cv2.COLOR_BGR2RGB)
        axes[i+1].imshow(roi_rgb)
        t = f"{det['final_label']}\nscore={det['final_score']:.2f} ({det['method']})"
        t += f"\nYOLO: {det['yolo_class']} ({det['yolo_conf']:.2f})"
        t += f"\nCLIP: {det['clip_label']} ({det['clip_score']:.2f})"
        t += f"\nsize: {det['bbox_size'][0]}x{det['bbox_size'][1]}px"
        axes[i+1].set_title(t, fontsize=8)
        axes[i+1].axis('off')

    plt.tight_layout()
    plt.show()

    for i, det in enumerate(detections):
        agree = 'AGREE' if det['yolo_class'] == det['clip_label'] else 'DISAGREE'
        print(f'  [{i+1}] {det["final_label"]} ({det["final_score"]:.2f}) '
              f'[{det["method"]}] bbox={det["bbox"]} size={det["bbox_size"]} '
              f'YOLO={det["yolo_class"]}({det["yolo_conf"]:.2f}) '
              f'CLIP={det["clip_label"]}({det["clip_score"]:.2f}) {agree}')


# ===== Test =====
print('\nTesting full pipeline v7.1...\n')
test_imgs = []
for d in [ARCHIVE_DIR/'split_dataset'/'test'/'images', ARCHIVE_DIR/'images', Path('data/full')]:
    if d.exists():
        imgs = list(d.rglob('*.jpg')) + list(d.rglob('*.png'))
        if imgs: test_imgs = imgs; break

random.seed(42)
sample = random.sample(test_imgs, min(8, len(test_imgs)))

# Stats
total_dets = 0
method_counts = defaultdict(int)
agree_count = 0

for img_path in sample:
    print(f'\n--- {img_path.name} ---')
    dets, img_bgr = detect_and_classify_v71(
        img_path, yolo_finetuned, classifier,
        preprocess_clip, class_names_list, device)
    visualize_v71(img_path, dets, img_bgr)
    total_dets += len(dets)
    for d in dets:
        method_counts[d['method']] += 1
        if d['yolo_class'] == d['clip_label']:
            agree_count += 1

print(f'\n===== PIPELINE STATS =====')
print(f'Total detections: {total_dets}')
print(f'YOLO-CLIP agreement: {agree_count}/{total_dets} ({agree_count/max(1,total_dets)*100:.0f}%)')
print(f'Method breakdown:')
for m, c in sorted(method_counts.items(), key=lambda x: -x[1]):
    print(f'  {m}: {c} ({c/max(1,total_dets)*100:.0f}%)')

FULL PIPELINE v7.1
  - YOLO detect -> Bounding Box
  - CLIP classify -> Label
  - Smart fusion khi YOLO va CLIP bat dong
  - Filter bbox qua nho
Class mapping: 38/52 YOLO classes co trong CLIP
  YOLO-only classes (CLIP khong biet): ['P.106a*Xe tải', 'P.117*', 'P.124a*', 'S.505a*Xe máy', 'S.505a*Xe tải và công', 'S.505a*Xe tải', 'Camera', 'P.127*50', 'P.127*60', 'P.127*80']...

Testing full pipeline v7.1...


--- 2620.jpg ---


<Figure size 1500x500 with 3 Axes>

  [1] I.434a (0.97) [clip] bbox=[1484, 458, 1539, 529] size=(np.int64(55), np.int64(71)) YOLO=I.434a(0.93) CLIP=I.434a(0.97) AGREE
  [2] R.415a (0.89) [clip] bbox=[991, 495, 1033, 531] size=(np.int64(42), np.int64(36)) YOLO=R.415a(0.35) CLIP=R.415a(0.89) AGREE

--- 0457.jpg ---


<Figure size 1500x500 with 3 Axes>

  [1] W.203c (0.63) [clip] bbox=[369, 154, 414, 192] size=(np.int64(45), np.int64(38)) YOLO=W.203c(0.87) CLIP=W.203c(0.63) AGREE
  [2] W.245a (1.00) [clip] bbox=[366, 190, 415, 229] size=(np.int64(49), np.int64(39)) YOLO=W.245a(0.87) CLIP=W.245a(1.00) AGREE

--- 0103.jpg ---


...

  [1] W.246a (0.96) [clip] bbox=[503, 51, 593, 124] size=(np.int64(90), np.int64(73)) YOLO=W.246a(0.92) CLIP=W.246a(0.96) AGREE
  [2] W.245a (1.00) [clip] bbox=[504, 0, 590, 47] size=(np.int64(86), np.int64(47)) YOLO=W.245a(0.89) CLIP=W.245a(1.00) AGREE

--- 0915.jpg ---


<Figure size 500x500 with 1 Axes>


--- 0572.jpg ---


<Figure size 500x500 with 1 Axes>


===== PIPELINE STATS =====
Total detections: 16
YOLO-CLIP agreement: 13/16 (81%)
Method breakdown:
  clip: 15 (94%)
  yolo_only: 1 (6%)


In [11]:
# CELL 10 - Save models + Summary

# Save CLIP classifier
torch.save({
    'state_dict': classifier.state_dict(),
    'class_names': class_names_list,
    'class_to_idx': class_to_idx,
    'clip_model': CLIP_MODEL,
    'clip_dim': CLIP_DIM,
    'num_classes': NUM_CLASSES,
    'p2_acc': p2_acc,
    'p3_acc': p3_acc,
}, 'clip_classifier_v7.pt')
print(f'CLIP classifier saved: clip_classifier_v7.pt')

# YOLO model already saved in runs/detect/traffic_signs_vn/weights/best.pt
print(f'YOLO model: {best_yolo_path}')

print(f'\n' + '=' * 60)
print('SUMMARY')
print('=' * 60)
print(f'  YOLO Detection:')
print(f'    mAP50: {metrics.box.map50:.3f}')
print(f'    mAP50-95: {metrics.box.map:.3f}')
print(f'  CLIP Classification:')
print(f'    Phase 2 (MLP Head): {p2_acc*100:.1f}%')
print(f'    Phase 3 (Fine-tune): {p3_acc*100:.1f}%')
print(f'  Full Pipeline: YOLO detect -> CLIP classify')
print(f'    CO BOUNDING BOX + LABEL + CONFIDENCE')
print('=' * 60)

CLIP classifier saved: clip_classifier_v7.pt
YOLO model: runs\detect\runs\detect\traffic_signs_vn_v2\weights\best.pt

SUMMARY
  YOLO Detection:
    mAP50: 0.989
    mAP50-95: 0.771
  CLIP Classification:
    Phase 2 (MLP Head): 95.1%
    Phase 3 (Fine-tune): 96.5%
  Full Pipeline: YOLO detect -> CLIP classify
    CO BOUNDING BOX + LABEL + CONFIDENCE


In [12]:
# ====================== CELL 11 MỚI - CHUẨN BỊ DATA BIỂN SỐ XE (FLAT DATASET) ======================
print('=' * 60)
print('MODULE B - BUOC 1: CHUAN BI DATA BIEN SO XE')
print('=' * 60)

from pathlib import Path
import zipfile
import shutil
import random
import yaml
import os

SEED = 42
random.seed(SEED)

PLATE_ZIP = Path('yolo_plate_dataset.zip')
PLATE_DIR = Path('plate_dataset')
PLATE_YOLO = None

print(f"📍 Thư mục hiện tại: {os.getcwd()}")

# ==================== GIẢI NÉN (nếu cần) ====================
if PLATE_ZIP.exists() and not PLATE_DIR.exists():
    print(f'📦 Giải nén {PLATE_ZIP}...')
    with zipfile.ZipFile(PLATE_ZIP, 'r') as z:
        z.extractall(PLATE_DIR)
    print(f'✅ Giải nén xong')
elif PLATE_DIR.exists():
    print(f'✅ Đã có {PLATE_DIR}')
else:
    print('❌ Không tìm thấy file zip!')

# ==================== TÌM SOURCE FOLDER (FLAT) ====================
src_folder = None

# Trường hợp 1: có thư mục yolo_plate_dataset (như dataset của bạn)
if (PLATE_DIR / 'yolo_plate_dataset').exists():
    src_folder = PLATE_DIR / 'yolo_plate_dataset'
    print(f'✅ Phát hiện dataset FLAT trong: {src_folder}')

# Trường hợp 2: file nằm trực tiếp trong plate_dataset
elif list(PLATE_DIR.glob('*.jpg')):
    src_folder = PLATE_DIR
    print(f'✅ Phát hiện dataset FLAT trực tiếp trong: {src_folder}')

if src_folder is None:
    print('❌ Không tìm thấy thư mục chứa ảnh + label!')
    PLATE_YOLO = None
else:
    # ==================== TẠO CẤU TRÚC YOLO ====================
    PLATE_YOLO = Path('plate_yolo_dataset')
    if PLATE_YOLO.exists():
        shutil.rmtree(PLATE_YOLO)
    
    for s in ['train', 'val']:
        (PLATE_YOLO / 'images' / s).mkdir(parents=True, exist_ok=True)
        (PLATE_YOLO / 'labels' / s).mkdir(parents=True, exist_ok=True)

    # ==================== THU THẬP TẤT CẢ FILE (FLAT) ====================
    all_imgs = sorted(list(src_folder.glob('*.jpg')) + 
                     list(src_folder.glob('*.png')) + 
                     list(src_folder.glob('*.jpeg')))
    
    pairs = []
    for img in all_imgs:
        lbl = src_folder / (img.stem + '.txt')
        if lbl.exists():
            pairs.append((img, lbl))

    print(f'🔍 Tìm thấy {len(pairs)} cặp ảnh + label')

    # ==================== TỰ SPLIT 80/20 ====================
    random.shuffle(pairs)
    n_train = int(len(pairs) * 0.8)

    print('📋 Đang copy file vào train/val (có thể mất 1-2 phút)...')

    for img, lbl in pairs[:n_train]:
        shutil.copy2(img, PLATE_YOLO / 'images' / 'train' / img.name)
        shutil.copy2(lbl, PLATE_YOLO / 'labels' / 'train' / lbl.name)
    
    for img, lbl in pairs[n_train:]:
        shutil.copy2(img, PLATE_YOLO / 'images' / 'val' / img.name)
        shutil.copy2(lbl, PLATE_YOLO / 'labels' / 'val' / lbl.name)

    # ==================== TẠO data.yaml ====================
    plate_yaml = {
        'path': str(PLATE_YOLO.resolve()),
        'train': 'images/train',
        'val': 'images/val',
        'nc': 1,
        'names': ['license_plate']
    }
    
    with open(PLATE_YOLO / 'data.yaml', 'w', encoding='utf-8') as f:
        yaml.dump(plate_yaml, f, default_flow_style=False, sort_keys=False)

    n_train = len(list((PLATE_YOLO / 'images' / 'train').glob('*')))
    n_val   = len(list((PLATE_YOLO / 'images' / 'val').glob('*')))

    print(f'\n🎉 HOÀN TẤT CHUẨN BỊ DATASET!')
    print(f'   Train images : {n_train}')
    print(f'   Val images   : {n_val}')
    print(f'   data.yaml    : {PLATE_YOLO}/data.yaml')
    print(f'   PLATE_YOLO   : {PLATE_YOLO.resolve()}')

MODULE B - BUOC 1: CHUAN BI DATA BIEN SO XE
📍 Thư mục hiện tại: c:\Đồ án\TrafficSignVN
✅ Đã có plate_dataset
✅ Phát hiện dataset FLAT trong: plate_dataset\yolo_plate_dataset
🔍 Tìm thấy 8259 cặp ảnh + label
📋 Đang copy file vào train/val (có thể mất 1-2 phút)...

🎉 HOÀN TẤT CHUẨN BỊ DATASET!
   Train images : 6607
   Val images   : 1652
   data.yaml    : plate_yolo_dataset/data.yaml
   PLATE_YOLO   : C:\Đồ án\TrafficSignVN\plate_yolo_dataset


In [14]:
# CELL 12 - FINE-TUNE YOLOv8s DETECT BIEN SO XE (UPGRADED)
from ultralytics import YOLO

print('=' * 60)
print('MODULE B: FINE-TUNE YOLOv8s DETECT BIEN SO XE')
print('  - YOLOv8s thay v8n')
print('  - imgsz=960, 100 epochs')
print('=' * 60)

yolo_plate_model = YOLO('yolov8s.pt')

print('Model: YOLOv8s | epochs=100 | imgsz=960 | batch=4')

plate_results = yolo_plate_model.train(
    data=str((PLATE_YOLO / 'data.yaml').resolve()),
    epochs=100,
    imgsz=960,
    batch=4,
    device=0,
    workers=0,
    patience=20,
    save=True,
    project='runs/detect',
    name='plate_detection_v2',
    exist_ok=True,

    # --- CÁC THAM SỐ KHẮC PHỤC LỖI ---
    deterministic=False, # Bắt buộc: Tắt deterministic để tránh lỗi ZeroDivisionError trong hàm interpolate của PyTorch
    multi_scale=False,   # Khuyên dùng: Tắt resize ngẫu nhiên để giữ độ phân giải cao 960 cho biển số nhỏ
    amp=True,            # Mixed precision (Giữ nguyên)
    
    # --- Augmentation cho bien so xe ---
    hsv_h=0.01,
    hsv_s=0.4,
    hsv_v=0.4,
    degrees=8,
    translate=0.15,
    scale=0.4,
    shear=2.0,
    perspective=0.001,      
    flipud=0.0,
    fliplr=0.5,
    mosaic=1.0,
    mixup=0.1,
    copy_paste=0.1,
    erasing=0.2,
    close_mosaic=20,

    # --- Optimizer & Learning Rate ---
    optimizer='AdamW',
    lr0=0.01,
    lrf=0.01,
    cos_lr=True,
    weight_decay=0.0005,
    warmup_epochs=5,
    cache=False,
)

print('\n>>> YOLO Bien So Xe v2 (YOLOv8s) HOAN TAT!')

MODULE B: FINE-TUNE YOLOv8s DETECT BIEN SO XE
  - YOLOv8s thay v8n
  - imgsz=960, 100 epochs
Model: YOLOv8s | epochs=100 | imgsz=960 | batch=4
New https://pypi.org/project/ultralytics/8.4.36 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.33  Python-3.11.9 torch-2.6.0+cu124 CUDA:0 (NVIDIA GeForce RTX 3070, 8192MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=4, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=20, cls=0.5, compile=False, conf=None, copy_paste=0.1, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=C:\ n\TrafficSignVN\plate_yolo_dataset\data.yaml, degrees=8, deterministic=False, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.2, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.01, hsv_s=0.4, hsv_v=0.4, imgsz=960, int8=False, iou=0.7, keras=False, kobj=1.0, lin

In [15]:
# CELL 13 - Danh gia YOLO bien so xe (voi TTA)
print('DANH GIA YOLO BIEN SO XE (YOLOv8s + TTA)')

plate_best = None
for pattern in ['plate_detection_v2', 'plate_detection']:
    for p in sorted(Path('runs/detect').rglob('best.pt')):
        if pattern in str(p):
            plate_best = p; break
    if plate_best: break

print(f'Loaded: {plate_best}')
yolo_plate_finetuned = YOLO(str(plate_best))

# Without TTA
print('\n--- Without TTA ---')
pm_no_tta = yolo_plate_finetuned.val(
    data=str((PLATE_YOLO / 'data.yaml').resolve()),
    workers=0, verbose=False)
print(f'  mAP50:    {pm_no_tta.box.map50:.4f}')
print(f'  mAP50-95: {pm_no_tta.box.map:.4f}')

# With TTA
print('\n--- With TTA ---')
plate_metrics = yolo_plate_finetuned.val(
    data=str((PLATE_YOLO / 'data.yaml').resolve()),
    workers=0, verbose=True, augment=True)
print(f'  mAP50:    {plate_metrics.box.map50:.4f}')
print(f'  mAP50-95: {plate_metrics.box.map:.4f}')

imp = (plate_metrics.box.map - pm_no_tta.box.map) * 100
print(f'  TTA improvement: +{imp:.2f}% mAP50-95')

# Visual test
plate_test = (list((PLATE_YOLO/'images'/'val').glob('*.jpg'))
             + list((PLATE_YOLO/'images'/'val').glob('*.png')))
if plate_test:
    random.seed(42)
    sample = random.sample(plate_test, min(6, len(plate_test)))
    fig, axes = plt.subplots(2, 3, figsize=(18, 12))
    axes = axes.flatten()
    for idx, ip in enumerate(sample[:6]):
        res = yolo_plate_finetuned(str(ip), conf=0.25, verbose=False, augment=True)
        img = cv2.cvtColor(cv2.imread(str(ip)), cv2.COLOR_BGR2RGB)
        boxes = res[0].boxes
        if boxes is not None:
            for j in range(len(boxes)):
                x1,y1,x2,y2 = [int(v) for v in boxes.xyxy[j].cpu().numpy()]
                conf = float(boxes.conf[j])
                cv2.rectangle(img, (x1,y1), (x2,y2), (0,255,0), 3)
                cv2.putText(img, f'{conf:.2f}', (x1,y1-5),
                           cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0,255,0), 2)
        axes[idx].imshow(img); axes[idx].axis('off')
        axes[idx].set_title(ip.name, fontsize=9)
    plt.suptitle('YOLOv8s Plate Detection (with TTA)', fontsize=14)
    plt.tight_layout(); plt.show()

DANH GIA YOLO BIEN SO XE (YOLOv8s + TTA)
Loaded: runs\detect\runs\detect\plate_detection_v2\weights\best.pt

--- Without TTA ---
Ultralytics 8.4.33  Python-3.11.9 torch-2.6.0+cu124 CUDA:0 (NVIDIA GeForce RTX 3070, 8192MiB)
Model summary (fused): 73 layers, 11,125,971 parameters, 0 gradients, 28.4 GFLOPs
val: Fast image access  (ping: 0.00.0 ms, read: 617.0399.1 MB/s, size: 30.6 KB)
val: Scanning C:\Đồ án\TrafficSignVN\plate_yolo_dataset\labels\val.cache... 1652 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 1652/1652  0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 104/104 4.9it/s 21.3s0.4s
                   all       1652       1690      0.994      0.995      0.995      0.735
Speed: 0.4ms preprocess, 7.0ms inference, 0.0ms loss, 0.5ms postprocess per image
Results saved to C:\ n\TrafficSignVN\runs\detect\val7
  mAP50:    0.9947
  mAP50-95: 0.7351

--- With TTA ---
Ultralytics 8.4.33  Python-3.11.9 torch-2.6.0+cu124

<Figure size 1800x1200 with 6 Axes>

In [16]:
# CELL 14 - FULL PIPELINE: BIEN BAO + BIEN SO XE + BOUNDING BOX
print('=' * 60)
print('FULL PIPELINE: BIEN BAO + BIEN SO XE')
print('=' * 60)

def clean_plate_text(text):
    cleaned = ''
    for ch in text.upper():
        if ch.isalnum() or ch in '.-': cleaned += ch
    return cleaned


@torch.no_grad()
def full_pipeline(image_path, yolo_sign, yolo_plate, clip_clf,
                  preprocess, clip_classes, device, conf=0.3):
    image_bgr = cv2.imread(str(image_path))
    if image_bgr is None:
        return {'signs': [], 'plates': [], 'image': None}

    h, w = image_bgr.shape[:2]

    # === A: Detect bien bao ===
    signs = []
    sr = yolo_sign(image_bgr, conf=conf, verbose=False)
    sboxes = sr[0].boxes
    if sboxes is not None:
        for j in range(len(sboxes)):
            x1,y1,x2,y2 = sboxes.xyxy[j].cpu().numpy().astype(int)
            yconf = float(sboxes.conf[j])
            ycls = int(sboxes.cls[j])
            yname = yolo_classes[ycls] if ycls < len(yolo_classes) else f'cls_{ycls}'
            bw, bh = x2-x1, y2-y1
            if bw < 20 or bh < 20: continue

            # CLIP classify
            pad = max(3, int(min(bw,bh)*0.1))
            roi = image_bgr[max(0,y1-pad):min(h,y2+pad), max(0,x1-pad):min(w,x2+pad)]
            if roi.shape[0] < 10 or roi.shape[1] < 10: continue
            clip_clf.eval()
            rpil = Image.fromarray(cv2.cvtColor(roi, cv2.COLOR_BGR2RGB))
            tensor = preprocess(rpil).unsqueeze(0).to(device)
            logits = clip_clf(tensor)
            probs = torch.softmax(logits, dim=1)[0]
            tv, ti = probs.topk(3)
            clip_label = clip_classes[ti[0]]
            clip_score = float(tv[0])
            top3 = [(clip_classes[ti[k]], float(tv[k])) for k in range(min(3, len(tv)))]

            final = clip_label if clip_score >= 0.4 else yname
            fscore = clip_score if clip_score >= 0.4 else yconf

            signs.append({
                'bbox': [int(x1),int(y1),int(x2),int(y2)],
                'label': final, 'score': fscore,
                'yolo_class': yname, 'yolo_conf': yconf,
                'clip_label': clip_label, 'clip_score': clip_score,
                'clip_top3': top3,
            })

    # === B: Detect bien so xe ===
    plates = []
    pr = yolo_plate(image_bgr, conf=conf, verbose=False)
    pboxes = pr[0].boxes
    if pboxes is not None:
        for j in range(len(pboxes)):
            x1,y1,x2,y2 = pboxes.xyxy[j].cpu().numpy().astype(int)
            pconf = float(pboxes.conf[j])
            bw, bh = x2-x1, y2-y1
            if bw < 15 or bh < 10: continue

            pad = max(3, int(min(bw,bh)*0.05))
            plate_crop = image_bgr[max(0,y1-pad):min(h,y2+pad), max(0,x1-pad):min(w,x2+pad)]
            if plate_crop.shape[0] < 10 or plate_crop.shape[1] < 15: continue

            # OCR
            gray = cv2.cvtColor(plate_crop, cv2.COLOR_BGR2GRAY)
            clahe = cv2.createCLAHE(clipLimit=3.0, tileGridSize=(8,8))
            gray = clahe.apply(gray)
            gray = cv2.bilateralFilter(gray, 9, 75, 75)

            # Thu OCR tren nhieu variant
            best_text, best_conf = '', 0.0
            for variant in [plate_crop, gray]:
                ocr_res = ocr_reader.readtext(variant, detail=1)
                for (_, text, conf) in ocr_res:
                    if conf > best_conf and len(text) >= 2:
                        best_text = text
                        best_conf = conf

            plate_text = clean_plate_text(best_text) if best_text else ''

            plates.append({
                'bbox': [int(x1),int(y1),int(x2),int(y2)],
                'plate_text': plate_text,
                'plate_raw': best_text,
                'plate_conf': best_conf,
                'detect_conf': pconf,
                'plate_crop': plate_crop,
            })

    return {'signs': signs, 'plates': plates, 'image': image_bgr}


def visualize_all(image_path, result):
    img_rgb = cv2.cvtColor(result['image'], cv2.COLOR_BGR2RGB).copy()

    # Ve bien bao (xanh la)
    for s in result['signs']:
        x1,y1,x2,y2 = s['bbox']
        cv2.rectangle(img_rgb, (x1,y1), (x2,y2), (0,255,0), 3)
        lbl = f"[Sign] {s['label']} {s['score']:.2f}"
        (tw,th),_ = cv2.getTextSize(lbl, cv2.FONT_HERSHEY_SIMPLEX, 0.55, 2)
        cv2.rectangle(img_rgb, (x1,max(0,y1-th-8)), (x1+tw+4,y1), (0,255,0), -1)
        cv2.putText(img_rgb, lbl, (x1+2,max(th,y1-3)),
                   cv2.FONT_HERSHEY_SIMPLEX, 0.55, (0,0,0), 2)

    # Ve bien so xe (do cam)
    for p in result['plates']:
        x1,y1,x2,y2 = p['bbox']
        cv2.rectangle(img_rgb, (x1,y1), (x2,y2), (255,80,0), 3)
        ptxt = p['plate_text'] if p['plate_text'] else '???'
        lbl = f"[Plate] {ptxt} ({p['plate_conf']:.2f})"
        (tw,th),_ = cv2.getTextSize(lbl, cv2.FONT_HERSHEY_SIMPLEX, 0.55, 2)
        cv2.rectangle(img_rgb, (x1,max(0,y1-th-8)), (x1+tw+4,y1), (255,80,0), -1)
        cv2.putText(img_rgb, lbl, (x1+2,max(th,y1-3)),
                   cv2.FONT_HERSHEY_SIMPLEX, 0.55, (255,255,255), 2)

    # Show
    n_roi = min(len(result['signs']) + len(result['plates']), 4)
    ncols = 1 + n_roi
    fig, axes = plt.subplots(1, max(ncols, 2), figsize=(5*max(ncols,2), 5))
    axes[0].imshow(img_rgb)
    ns, np_ = len(result['signs']), len(result['plates'])
    axes[0].set_title(f"{Path(image_path).name}\n{ns} sign(s) | {np_} plate(s)", fontsize=11)
    axes[0].axis('off')

    roi_idx = 1
    # Show sign ROIs
    for s in result['signs'][:2]:
        if roi_idx >= len(axes): break
        x1,y1,x2,y2 = s['bbox']
        roi = result['image'][y1:y2, x1:x2]
        if roi.size > 0:
            axes[roi_idx].imshow(cv2.cvtColor(roi, cv2.COLOR_BGR2RGB))
            axes[roi_idx].set_title(f"Sign: {s['label']}\n{s['score']:.2f}", fontsize=9)
            axes[roi_idx].axis('off')
            roi_idx += 1

    # Show plate ROIs
    for p in result['plates'][:2]:
        if roi_idx >= len(axes): break
        if p['plate_crop'] is not None and p['plate_crop'].size > 0:
            axes[roi_idx].imshow(cv2.cvtColor(p['plate_crop'], cv2.COLOR_BGR2RGB))
            ptxt = p['plate_text'] if p['plate_text'] else '???'
            axes[roi_idx].set_title(f"Plate: {ptxt}\n{p['plate_conf']:.2f}", fontsize=9)
            axes[roi_idx].axis('off')
            roi_idx += 1

    for k in range(roi_idx, len(axes)):
        axes[k].axis('off')

    plt.tight_layout(); plt.show()

    # Print
    if result['signs']:
        print('  BIEN BAO:')
        for i, s in enumerate(result['signs']):
            print(f'    {i+1}. {s["label"]} (score={s["score"]:.2f})'
                  f' | YOLO={s["yolo_class"]}({s["yolo_conf"]:.2f})'
                  f' CLIP={s["clip_label"]}({s["clip_score"]:.2f})')
            print(f'       Top3: {s["clip_top3"]}')
    if result['plates']:
        print('  BIEN SO XE:')
        for i, p in enumerate(result['plates']):
            ptxt = p['plate_text'] if p['plate_text'] else 'N/A'
            print(f'    {i+1}. "{ptxt}" (conf={p["plate_conf"]:.2f},'
                  f' detect={p["detect_conf"]:.2f}) raw="{p["plate_raw"]}"')


# === TEST ===
print('\nTesting full pipeline (bien bao + bien so)...\n')
test_imgs = []
for d in [ARCHIVE_DIR/'images', Path('data/full'), ARCHIVE_DIR/'split_dataset'/'test'/'images']:
    if d.exists():
        imgs = list(d.rglob('*.jpg')) + list(d.rglob('*.png'))
        if imgs: test_imgs = imgs; break

if test_imgs:
    random.seed(42)
    for img_path in random.sample(test_imgs, min(8, len(test_imgs))):
        print(f'\n--- {img_path.name} ---')
        result = full_pipeline(
            img_path, yolo_finetuned, yolo_plate_finetuned,
            classifier, preprocess_clip, class_names_list, device)
        visualize_all(img_path, result)
else:
    print('Khong tim thay anh test')

FULL PIPELINE: BIEN BAO + BIEN SO XE

Testing full pipeline (bien bao + bien so)...


--- 2620.jpg ---


<Figure size 2500x500 with 5 Axes>

  BIEN BAO:
    1. I.434a (score=0.97) | YOLO=I.434a(0.93) CLIP=I.434a(0.97)
       Top3: [('I.434a', 0.9662730693817139), ('I.407a', 0.009338336065411568), ('W.246a', 0.004733200650662184)]
    2. P.137 (score=0.99) | YOLO=P.137(0.78) CLIP=P.137(0.99)
       Top3: [('P.137', 0.9947570562362671), ('P.123b', 0.0016171608585864305), ('P.124b1', 0.0011061818804591894)]
    3. R.415a (score=0.97) | YOLO=R.415a(0.35) CLIP=R.415a(0.97)
       Top3: [('R.415a', 0.9685709476470947), ('P.130', 0.0052847289480268955), ('I.407a', 0.005042937584221363)]
  BIEN SO XE:
    1. "0607" (conf=0.21, detect=0.77) raw="06*07"
    2. "BU" (conf=0.94, detect=0.72) raw="bu"
    3. "N/A" (conf=0.00, detect=0.69) raw=""
    4. "N/A" (conf=0.00, detect=0.49) raw=""
    5. "N/A" (conf=0.00, detect=0.39) raw=""
    6. "N/A" (conf=0.00, detect=0.38) raw=""
    7. "N/A" (conf=0.00, detect=0.32) raw=""

--- 0457.jpg ---


<Figure size 2500x500 with 5 Axes>

  BIEN BAO:
    1. W.203c (score=0.81) | YOLO=W.203c(0.87) CLIP=W.203c(0.81)
       Top3: [('W.203c', 0.8052017688751221), ('P.123b', 0.040380947291851044), ('W.246a', 0.03079703450202942)]
    2. W.245a (score=1.00) | YOLO=W.245a(0.87) CLIP=W.245a(1.00)
       Top3: [('W.245a', 0.9988366961479187), ('W.202b', 0.00020953141211066395), ('P.124c', 0.000180947725311853)]
  BIEN SO XE:
    1. "N/A" (conf=0.00, detect=0.65) raw=""
    2. "N/A" (conf=0.00, detect=0.45) raw=""
    3. "N/A" (conf=0.00, detect=0.43) raw=""
    4. "N/A" (conf=0.00, detect=0.33) raw=""

--- 0103.jpg ---


...

  BIEN BAO:
    1. W.246a (score=0.97) | YOLO=W.246a(0.92) CLIP=W.246a(0.97)
       Top3: [('W.246a', 0.970848560333252), ('P.137', 0.0060378327034413815), ('W.207', 0.004063682164996862)]
    2. W.245a (score=1.00) | YOLO=W.245a(0.89) CLIP=W.245a(1.00)
       Top3: [('W.245a', 0.9985484480857849), ('R.302a', 0.000419700110796839), ('P.139', 0.0004015064623672515)]
  BIEN SO XE:
    1. "ICHUYQUANSAT" (conf=0.15, detect=0.56) raw="Ichu Y QUaN SAt |"

--- 0915.jpg ---


<Figure size 2500x500 with 5 Axes>

  BIEN BAO:
    1. P.123a (score=0.98) | YOLO=P.123a(0.83) CLIP=P.123a(0.98)
       Top3: [('P.123a', 0.9755247831344604), ('R.301e', 0.01566610112786293), ('P.124b1', 0.0009386939345858991)]
  BIEN SO XE:
    1. "N/A" (conf=0.00, detect=0.74) raw=""
    2. "N/A" (conf=0.00, detect=0.59) raw=""
    3. "N/A" (conf=0.00, detect=0.40) raw=""
    4. "N/A" (conf=0.00, detect=0.38) raw=""

--- 0572.jpg ---


<Figure size 2500x500 with 5 Axes>

  BIEN BAO:
    1. P.117* (score=0.93) | YOLO=P.117*(0.93) CLIP=P.129(0.16)
       Top3: [('P.129', 0.16173645853996277), ('W.203c', 0.11321523785591125), ('B.8a', 0.10292433947324753)]
    2. P.127*40 (score=0.92) | YOLO=P.127*40(0.92) CLIP=P.123a(0.10)
       Top3: [('P.123a', 0.10148673504590988), ('P.107', 0.09242742508649826), ('B.8a', 0.08595331758260727)]
    3. P.107 (score=0.89) | YOLO=P.107(0.89) CLIP=P.107(0.35)
       Top3: [('P.107', 0.3547864556312561), ('B.8a', 0.23338067531585693), ('P.103a', 0.08724259585142136)]
    4. P.107 (score=0.44) | YOLO=P.107(0.87) CLIP=P.107(0.44)
       Top3: [('P.107', 0.4361334443092346), ('B.8a', 0.24709393084049225), ('P.123a', 0.04878978058695793)]
    5. P.130 (score=0.84) | YOLO=P.130(0.86) CLIP=P.130(0.84)
       Top3: [('P.130', 0.8444468975067139), ('P.137', 0.01781192049384117), ('R.415a', 0.01231481321156025)]
    6. P.130 (score=0.50) | YOLO=P.130(0.86) CLIP=P.130(0.50)
       Top3: [('P.130', 0.5019378662109375), ('W.203c', 0.0

In [17]:
# CELL 15 - Save + Summary

torch.save({
    'state_dict': classifier.state_dict(),
    'class_names': class_names_list,
    'class_to_idx': class_to_idx,
    'clip_model': CLIP_MODEL,
    'clip_dim': CLIP_DIM,
    'num_classes': NUM_CLASSES,
    'p2_acc': p2_acc,
    'p3_acc': p3_acc,
}, 'clip_classifier_v7.pt')

print('MODELS SAVED:')
print(f'  1. CLIP classifier: clip_classifier_v7.pt')
print(f'  2. YOLO bien bao (v8s): {best_yolo_path}')
print(f'  3. YOLO bien so (v8s):  {plate_best}')

print(f'\n{"="*60}')
print('TONG KET DO AN (UPGRADED)')
print(f'{"="*60}')
print(f'  MODULE A - BIEN BAO GIAO THONG:')
print(f'    Model:    YOLOv8s (11.2M params)')
print(f'    mAP50:    {metrics.box.map50:.4f}')
print(f'    mAP50-95: {metrics.box.map:.4f}  (with TTA)')
print(f'    CLIP:     {p3_acc*100:.1f}% accuracy')
print(f'')
print(f'  MODULE B - BIEN SO XE:')
print(f'    Model:    YOLOv8s (11.2M params)')
print(f'    mAP50:    {plate_metrics.box.map50:.4f}')
print(f'    mAP50-95: {plate_metrics.box.map:.4f}  (with TTA)')
print(f'    OCR:      EasyOCR')
print(f'')
print(f'  KY THUAT TANG mAP50-95:')
print(f'    1. YOLOv8s thay v8n (3x params)')
print(f'    2. imgsz=960 (thay 640)')
print(f'    3. 100 epochs (thay 50)')
print(f'    4. Multi-scale training')
print(f'    5. TTA inference')
print(f'    6. Copy-paste + mixup + perspective aug')
print(f'    7. Cosine LR + longer warmup')
print(f'{"="*60}')

MODELS SAVED:
  1. CLIP classifier: clip_classifier_v7.pt
  2. YOLO bien bao (v8s): runs\detect\runs\detect\traffic_signs_vn_v2\weights\best.pt
  3. YOLO bien so (v8s):  runs\detect\runs\detect\plate_detection_v2\weights\best.pt

TONG KET DO AN (UPGRADED)
  MODULE A - BIEN BAO GIAO THONG:
    Model:    YOLOv8s (11.2M params)
    mAP50:    0.9891
    mAP50-95: 0.7710  (with TTA)
    CLIP:     96.5% accuracy

  MODULE B - BIEN SO XE:
    Model:    YOLOv8s (11.2M params)
    mAP50:    0.9946
    mAP50-95: 0.7212  (with TTA)
    OCR:      EasyOCR

  KY THUAT TANG mAP50-95:
    1. YOLOv8s thay v8n (3x params)
    2. imgsz=960 (thay 640)
    3. 100 epochs (thay 50)
    4. Multi-scale training
    5. TTA inference
    6. Copy-paste + mixup + perspective aug
    7. Cosine LR + longer warmup
